# Schema Validation

1. What is Schema-on-read / Schema-on-write? 
2. Column order validation
3. Data Type validation
4. Column name validation
5. Nullability validation
6. Extra columns validation

In [ ]:
%load_ext sql

In [1]:
%sql spark

In [2]:
df = spark.read.format("parquet").load("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df.printSchema()

#
display(df)

# Save as delta table into Minio S3
#df.write.format('delta').save('s3a://warehouse/default/deltalake/invoices_sv')

root
 |-- customer_id: integer (nullable = true)
 |-- invoice_no: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- shopping_mall: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

+-----------+----------+------+---+---------------+--------+-------+--------------+------------+----------------+-------------+
|customer_id|invoice_no|gender|age|category       |quantity|price  |payment_method|invoice_date|shopping_mall   |_rescued_data|
+-----------+----------+------+---+---------------+--------+-------+--------------+------------+----------------+-------------+
|1          |I178410   |Male  |61 |Clothing       |5       |1500.4 |Credit Card   |2021-11-26  |Metrocity       |NULL         |
|2          |I158163   |Ma

In [ ]:
%%sql

DROP TABLE IF EXISTS spark_catalog.deltalake.invoices_sv;
--DROP TABLE IF EXISTS delta.`s3a://warehouse/default/deltalake/invoices_sv`;

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/invoices_sv --recursive

##### Create Delta Table

In [ ]:
%%sql

show databases;


In [ ]:
%%sql

CREATE OR REPLACE TABLE spark_catalog.deltalake.invoices_sv(
  customer_id INT NOT NULL,  
  invoice_no STRING,
  quantity INT,
  price FLOAT, 
  invoice_date DATE
) USING DELTA; 



In [ ]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_sv
SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`; 


In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_sv`
LIMIT 5;

In [ ]:
%%sql

SELECT MIN(customer_id), MAX(customer_id), COUNT(*)
FROM delta.`s3a://warehouse/default/deltalake/invoices_sv`;

#### Scenario 1: Column Order Validation

In [ ]:
%%sql

INSERT INTO delta.`s3a://warehouse/default/deltalake/invoices_sv`
SELECT quantity, invoice_no, customer_id, price, invoice_date
FROM VALUES (99999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_sv`
WHERE customer_id = 99999;

In [ ]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_sv
WHERE customer_id = 10;

In [ ]:
%%sql

SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
ORDER BY customer_id DESC LIMIT 5;

In [ ]:
%%sql

MERGE INTO spark_catalog.deltalake.invoices_sv AS tgt
USING (
  SELECT quantity, invoice_no, customer_id, CAST (price AS FLOAT), invoice_date
  FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
  ORDER BY customer_id DESC LIMIT 5
) src 
ON tgt.customer_id = src.customer_id
WHEN MATCHED THEN 
  UPDATE SET 
    tgt.customer_id = src.customer_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.invoice_date = current_date
WHEN NOT MATCHED THEN
  INSERT * 

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_sv`
WHERE customer_id > 100; 

#### Scenario 2: Data Type Validation

In [ ]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_sv
VALUES (
  'ABC', 
  'I45678',
  10,
  98.75,
  '2025-01-01'
)

In [ ]:
%%sql

INSERT INTO delta.`s3a://warehouse/default/deltalake/invoices_sv`
VALUES (
  '99499', 
  'I45678',
  10,
  98.75,
  '2025-01-01'
)

In [ ]:
%%sql

select * from spark_catalog.deltalake.invoices_sv where customer_id = 99499;

#### Scenario 3: Column Name Validation

column validate by position

In [ ]:
%%sql

-- column customer_id remaned as c_id & quantity as qty

INSERT INTO delta.`s3a://warehouse/default/deltalake/invoices_sv`
SELECT customer_id AS c_id, invoice_no, quantity AS qty, price, invoice_date
FROM VALUES (99999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_sv`
WHERE customer_id = 99999;

In [ ]:
%%sql

SELECT customer_id, invoice_no, quantity, CAST (price AS FLOAT), invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
ORDER BY customer_id LIMIT 5

In [ ]:
%%sql

-- it will fail because merge update by column name 

MERGE INTO delta.`s3a://warehouse/default/deltalake/invoices_sv` AS tgt
USING (
  SELECT customer_id AS c_id, invoice_no, quantity AS qty, CAST (price AS FLOAT), invoice_date
  FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
  ORDER BY customer_id LIMIT 5
) src 
ON tgt.customer_id = src.c_id
WHEN MATCHED THEN 
  UPDATE SET 
    tgt.customer_id = src.c_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.qty,
    tgt.price = src.price,
    tgt.invoice_date = current_date
WHEN NOT MATCHED THEN
  INSERT * 

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_sv`
WHERE customer_id > 100;

#### Scenario 4: Nullability Validation

In [ ]:
%%sql

INSERT INTO delta.`s3a://warehouse/default/deltalake/invoices_sv`
VALUES (NULL, NULL, NULL, NULL, NULL);

In [ ]:
%%sql

INSERT INTO delta.`s3a://warehouse/default/deltalake/invoices_sv`
VALUES (78912, NULL, NULL, NULL, NULL);

#### Scenario 5: Extra Column Validation

In [ ]:
%%sql

-- new extra column customer_type added
INSERT INTO delta.`s3a://warehouse/default/deltalake/invoices_sv`
SELECT customer_id, invoice_no, quantity, price, invoice_date, "VIP" AS customer_type
FROM VALUES (78999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

In [ ]:
%%sql

SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
WHERE customer_id BETWEEN 150 AND 155

In [ ]:
%%sql

MERGE INTO delta.`s3a://warehouse/default/deltalake/invoices_sv` AS tgt
USING (
  SELECT customer_id, invoice_no, quantity, price, invoice_date, "VIP" AS customer_type
  FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
  WHERE customer_id BETWEEN 150 AND 155
) src 
ON tgt.customer_id = src.customer_id
WHEN NOT MATCHED THEN
  INSERT * 

In [ ]:
%%sql

SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM delta.`s3a://warehouse/default/deltalake/invoices_sv`
WHERE customer_id BETWEEN 150 AND 155

In [ ]:
%%sql

DESC HISTORY spark_catalog.deltalake.invoices_sv;


In [ ]:
display(spark.sql("""DESC HISTORY spark_catalog.deltalake.invoices_sv"""));

In [ ]:
%%sql 

DESC DETAIL spark_catalog.deltalake.invoices_sv;


In [ ]:
display(spark.sql("""DESC DETAIL spark_catalog.deltalake.invoices_sv"""))